# Cuaderno 00 · Inventario y trazabilidad del expediente

**Expediente:** BES-jun · **Estado del expediente:** Listo para revisión (no Completo)

> **Relación con `00_inventario_y_fuentes`.** Ese cuaderno responde *¿qué insumo se recibió y está intacto?* (huella del libro y de las fuentes técnicas). Este responde *¿qué contiene el expediente, quién produjo cada archivo y en qué estado está cada cuaderno?* Son dos preguntas distintas y por eso son dos cuadernos.

## 1. Introducción y contexto (el problema)
Un expediente de análisis crece: cuadernos, módulos de código, tablas intermedias, figuras, documentos y PDF. Con el tiempo aparecen tres riesgos: archivos huérfanos (nadie sabe quién los produjo), archivos desactualizados (se generaron con una versión anterior del análisis) y cuadernos que dicen una cosa mientras el código hace otra. En un entorno donde una cifra obsoleta puede llegar a una decisión de operación, la falta de trazabilidad es un riesgo de integridad.

## 2. Justificación y necesidad
Un inventario verificable de los artefactos, un mapa de **linaje** (cuaderno → archivos que produce), el estado real de ejecución de los 12 cuadernos y una comparación con la versión original respaldada antes de la auditoría. El proyecto **no es un repositorio Git**, de modo que el respaldo con manifiesto SHA-256 es la única referencia verificable para saber qué cambió.

## 3. Objetivo general
Dejar documentado qué hay en el expediente, cómo se relacionan los 12 cuadernos entre sí y con los archivos que producen, y demostrar que los originales respaldados no han sido alterados.

## 4. Dominio y alcance
**Incluye:** inventario por carpeta; roles y relación de los 12 cuadernos (incluidos los de numeración repetida, que **no** son duplicados); linaje declarado con verificación de existencia y huella; comparación con el respaldo previo; estado de ejecución leído de los `.ipynb` en disco.

**No incluye:** validar el contenido técnico de los cuadernos (eso es cada cuaderno), ni mover, borrar o renumerar archivos.

## 5. Fuentes y datos (entradas y salidas)
**Entradas:** la carpeta del expediente, los 12 `.ipynb`, el respaldo `logs/respaldo_previo_a_auditoria/` con `MANIFIESTO_SHA256.txt`. **Salidas:** tablas de inventario, linaje y estado; ningún archivo escrito.

## 6. Objetivos específicos y criterios de aceptación
**C00b-1** el respaldo coincide con su manifiesto (24 archivos) · **C00b-2** existen los 12 cuadernos esperados · **C00b-3** todo archivo declarado en el linaje existe · **C00b-4** los roles de los 12 cuadernos son distintos (ninguno es duplicado de otro) · **C00b-5** cada cuaderno (salvo éste) tiene Limitaciones y Referencias · **C00b-6** el libro no cambió.

## 7. Riesgo si este paso falla
Usar tablas o figuras generadas con una versión anterior sin saberlo, o creer que se conservan los originales cuando se han sobrescrito.

## Preparación del entorno
Misma celda de los demás cuadernos: localiza el expediente, fija formato de tablas y muestra la ficha del entorno (versiones y SHA-256 del insumo).

In [1]:
from pathlib import Path
import os, sys

def localizar_expediente() -> Path:
    # Sube desde el directorio actual hasta la carpeta que tiene dataset.md y entrada/.
    for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (p / "dataset.md").is_file() and (p / "entrada").is_dir():
            return p
    raise RuntimeError("Ejecute el cuaderno dentro del expediente (donde están dataset.md y entrada/).")

RAIZ = localizar_expediente()                 # .../datasets/en_uso/BES-jun
RAIZ_PROYECTO = RAIZ.parents[2]               # .../HUC
os.chdir(RAIZ)
sys.path.insert(0, str(RAIZ / "src"))

import pandas as pd
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 180)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.float_format", lambda v: f"{v:,.8g}")

from analisis.criterios import Criterios
from analisis.entorno import ficha
from analisis.presentacion import T
T(pd.DataFrame(list(ficha().items()), columns=["campo", "valor"]))

campo,valor
fecha_ejecucion,2026-10-20T14:26:15-03:00
python,3.14.8
sistema,Linux 7.2.10-100.fc43.x86_64
pandas,3.0.6
numpy,2.5.4
matplotlib,3.11.2
openpyxl,3.1.5
expediente,BES-jun
insumo,entrada/BES_jun.xlsx
insumo_sha256,ccb5a738baea13321993e8b6eccefd6cac90ecda86b1c4b9181f897fcfa83f85


## Paso 1 · Inventario por carpeta

**Qué se hace.** Se recorre el expediente (sin `__pycache__` ni `.ipynb_checkpoints`) y se cuenta, por carpeta, cuántos archivos hay y cuánto ocupan. **Para qué.** Para tener una fotografía de la composición del expediente; una carpeta inesperadamente vacía o con archivos de más es una señal.

In [2]:
from ingesta.integridad import huellas
from ingesta.leer_bes_jun import COPIA_TRABAJO
sha_inicial = huellas(COPIA_TRABAJO)["sha256"]
IGNORAR = {"__pycache__", ".ipynb_checkpoints", ".pytest_cache"}
archivos = [p for p in RAIZ.rglob("*") if p.is_file() and not (set(p.relative_to(RAIZ).parts) & IGNORAR)]
inv = pd.DataFrame([{"carpeta": p.relative_to(RAIZ).parts[0] if len(p.relative_to(RAIZ).parts) > 1 else "(raíz)", "bytes": p.stat().st_size} for p in archivos])
resumen = inv.groupby("carpeta").agg(archivos=("bytes", "size"), kilobytes=("bytes", lambda s: s.sum() / 1024)).reset_index()
print("Total de archivos:", len(archivos))
T(resumen)

Total de archivos: 150


carpeta,archivos,kilobytes
(raíz),3,2.5546875
data_procesados,14,168.68262
docs,4,21.881836
entrada,1,321.58984
logs,28,1201.2764
notebooks,12,1458.8477
pdf,25,1194.7725
reports,12,488.18164
src,47,384.78711
tests,4,18.214844


**Interpretación.** El expediente se reparte entre `entrada/` (el insumo), `src/` (código reutilizable), `notebooks/` (los 12 cuadernos), `data_procesados/` (tablas derivadas), `reports/` (informe y figuras), `pdf/` (documentos de entrega), `docs/` (decisiones y consultas), `tests/` (pruebas) y `logs/` (bitácora y respaldo). La separación es la que permite aplicar una regla simple: **`entrada/` solo se lee; todo lo demás se puede regenerar a partir de ella y del código**.

## Paso 2 · Los 12 cuadernos: quién es quién

**Hecho importante.** Cinco números (00 a 04) tienen **dos** cuadernos cada uno. **No son duplicados**: cada par responde preguntas diferentes sobre el mismo tema. Se declara aquí el rol de cada uno y se comprueba que ningún rol se repite. Para cada cuaderno se lee del disco su estructura real (celdas, palabras explicativas, criterios `C…` encontrados y estado de ejecución).

In [3]:
import nbformat, re
ROLES = [
    ("00_inventario_y_fuentes",               "Identidad del insumo: huellas del libro y de las fuentes técnicas; control negativo."),
    ("00_inventario_y_trazabilidad",          "Inventario del expediente, linaje de archivos y estado de los 12 cuadernos (este cuaderno)."),
    ("01_inspeccion_BES_jun",                 "Inspección descriptiva de las hojas: mapa de bloques, dimensiones, entradas, survey y equipos."),
    ("01_inspeccion_y_calidad_datos",         "Calidad: controles de errores, fechas, survey, curvatura mínima, TVD y plausibilidad."),
    ("02_ecuaciones_y_variables",             "Glosario de ecuaciones y variables: fenómeno, unidades, hipótesis, función en código y categoría."),
    ("02_trazabilidad_y_analisis_formulas",   "Trazabilidad de fórmulas del libro: firmas, constantes, hidrostática y catálogo."),
    ("03_limpieza_y_mapeo",                   "Mapeo del formulario a tabla larga y prueba de ida y vuelta; registro de unidades."),
    ("03_mapeo_y_preparacion_BES_jun",        "Preparación de tablas ordenadas de survey y curvas de bomba; estadísticas y figuras."),
    ("04_calculos_deterministicos",           "Contrato y verificación de las funciones de cálculo; casos límite y ficha de confiabilidad."),
    ("04_modelo_deterministico",              "Formulación del modelo, dominio de validez y sensibilidad local."),
    ("05_evaluacion_por_pozo",                "Evaluación por pozo: reproducción, contraste, ventana operativa, sensibilidad a ambigüedades."),
    ("06_hallazgos_y_reglas_candidatas",      "Registro de hallazgos, consultas técnicas y reglas candidatas."),
]
filas = []
for i, (nombre, rol) in enumerate(ROLES):
    ruta = RAIZ / "notebooks" / f"{nombre}.ipynb"
    if not ruta.is_file():
        filas.append({"n.º": i, "cuaderno": nombre, "rol": rol, "existe": False}); continue
    nb = nbformat.read(ruta, as_version=4)
    md = [c for c in nb.cells if c.cell_type == "markdown"]; co = [c for c in nb.cells if c.cell_type == "code"]
    texto = "\n".join(c.source for c in nb.cells)
    filas.append({"n.º": i, "cuaderno": nombre, "existe": True, "celdas MD": len(md), "celdas de código": len(co),
                  "palabras explicativas": sum(len(c.source.split()) for c in md),
                  "criterios C…": len(set(re.findall(r"\*\*(C\d+b?-\d+)\*\*", texto))),
                  "con salidas guardadas": sum(1 for c in co if c.outputs), "con Limitaciones y Referencias": "## Limitaciones" in texto and "## Referencias" in texto})
estado = pd.DataFrame(filas)
T(estado, ancha=True)

n.º,cuaderno,existe,celdas MD,celdas de código,palabras explicativas,criterios C…,con salidas guardadas,con Limitaciones y Referencias
0,00_inventario_y_fuentes,sí,15,10,1630,6,10,sí
1,00_inventario_y_trazabilidad,sí,14,9,1467,6,0,sí
2,01_inspeccion_BES_jun,sí,16,13,1971,6,13,sí
3,01_inspeccion_y_calidad_datos,sí,22,16,2218,9,16,sí
4,02_ecuaciones_y_variables,sí,16,7,2854,6,7,sí
5,02_trazabilidad_y_analisis_formulas,sí,19,13,2109,5,13,sí
6,03_limpieza_y_mapeo,sí,22,13,2003,6,12,sí
7,03_mapeo_y_preparacion_BES_jun,sí,19,13,2010,7,13,sí
8,04_calculos_deterministicos,sí,17,13,1563,6,12,sí
9,04_modelo_deterministico,sí,18,11,2385,6,11,sí


In [4]:
T(pd.DataFrame(ROLES, columns=["cuaderno", "rol (qué pregunta responde)"]), ancha=True)

cuaderno,rol (qué pregunta responde)
00_inventario_y_fuentes,Identidad del insumo: huellas del libro y de las fuentes técnicas; control negativo.
00_inventario_y_trazabilidad,"Inventario del expediente, linaje de archivos y estado de los 12 cuadernos (este cuaderno)."
01_inspeccion_BES_jun,"Inspección descriptiva de las hojas: mapa de bloques, dimensiones, entradas, survey y equipos."
01_inspeccion_y_calidad_datos,"Calidad: controles de errores, fechas, survey, curvatura mínima, TVD y plausibilidad."
02_ecuaciones_y_variables,"Glosario de ecuaciones y variables: fenómeno, unidades, hipótesis, función en código y categoría."
02_trazabilidad_y_analisis_formulas,"Trazabilidad de fórmulas del libro: firmas, constantes, hidrostática y catálogo."
03_limpieza_y_mapeo,Mapeo del formulario a tabla larga y prueba de ida y vuelta; registro de unidades.
03_mapeo_y_preparacion_BES_jun,Preparación de tablas ordenadas de survey y curvas de bomba; estadísticas y figuras.
04_calculos_deterministicos,Contrato y verificación de las funciones de cálculo; casos límite y ficha de confiabilidad.
04_modelo_deterministico,"Formulación del modelo, dominio de validez y sensibilidad local."


**Interpretación.** Cada cuaderno tiene un rol propio y no hay dos con la misma pregunta. Los pares de numeración repetida se relacionan así: el primero de cada par (`01_inspeccion_BES_jun`, `02_ecuaciones_y_variables`, `03_mapeo_y_preparacion_BES_jun`, `04_modelo_deterministico`) **describe y formula**; el segundo (`01_inspeccion_y_calidad_datos`, `02_trazabilidad_y_analisis_formulas`, `03_limpieza_y_mapeo`, `04_calculos_deterministicos`) **verifica y audita**. Se conservan ambos porque fusionarlos mezclaría descripción con evidencia de control, y porque renumerar o eliminar uno rompería referencias cruzadas existentes sin aportar nada técnico.

La columna «con salidas guardadas» indica cuántas celdas de código de cada cuaderno traen salida en el archivo; una cifra de cero en un cuaderno con código indicaría que no se ejecutó. **Esto lee el disco en el momento de esta ejecución**: el orquestador ejecuta este cuaderno al final del lote para que refleje el estado definitivo.

## Paso 3 · Linaje: qué archivo produce cada cuaderno

**Qué se hace.** Se declara, en una tabla, qué archivos de `data_procesados/` y `reports/figuras/` produce cada cuaderno y se verifica que existan y tengan tamaño positivo; se registra también su SHA-256. **Nota:** `guardar_figuras` regenera siempre las seis figuras (es determinista); cada cuaderno se declara productor de las que analiza. **Límite honesto:** el linaje es **declarado**; el control comprueba existencia y huella, no que el cuaderno indicado haya sido realmente el que lo escribió. Los archivos del directorio que no figuran en el linaje se listan aparte: son los «huérfanos», una señal de higiene del expediente.

In [5]:
LINAJE = {
    "03_limpieza_y_mapeo": ["data_procesados/variables_largo.csv", "data_procesados/matriz_entradas_por_hoja.csv", "data_procesados/registro_unidades.csv"],
    "01_inspeccion_BES_jun": ["reports/figuras/survey_md_tvd_por_hoja.png", "reports/figuras/inclinacion_survey_por_hoja.png"],
    "03_mapeo_y_preparacion_BES_jun": ["data_procesados/curvas_bomba_60hz.csv", "reports/figuras/head_puntos_60hz.png"],
    "04_calculos_deterministicos": ["data_procesados/resultados_deterministicos.csv"],
    "05_evaluacion_por_pozo": ["reports/figuras/caudal_limites_por_hoja.png", "reports/figuras/interseccion_sistema_bomba.png", "reports/figuras/impacto_ambiguedades_en_caudal.png"],
}
filas = []
for cuaderno, rutas in LINAJE.items():
    for r in rutas:
        p = RAIZ / r
        filas.append({"cuaderno productor": cuaderno, "archivo": r, "existe": p.is_file() and p.stat().st_size > 0,
                      "bytes": p.stat().st_size if p.is_file() else None, "SHA-256": huellas(p)["sha256"] if p.is_file() else None})
linaje = pd.DataFrame(filas)
declarados = {RAIZ / r for rs in LINAJE.values() for r in rs}
presentes = {p for p in (RAIZ / "data_procesados").glob("*") if p.is_file()} | {p for p in (RAIZ / "reports" / "figuras").glob("*") if p.is_file()}
huerfanos = sorted(str(p.relative_to(RAIZ)) for p in presentes - declarados)
print("Archivos declarados:", len(declarados), "| presentes sin declarar:", len(huerfanos))
T(linaje, ancha=True)

Archivos declarados: 11 | presentes sin declarar: 11


cuaderno productor,archivo,existe,bytes,SHA-256
03_limpieza_y_mapeo,data_procesados/variables_largo.csv,sí,17595,74afaf60ba3c551853d1f1e6527aa9afd7730566c995c4022cc72c13ce6dcbed
03_limpieza_y_mapeo,data_procesados/matriz_entradas_por_hoja.csv,sí,2495,7fec963dfeacdc7a9ddfd7e0f3fa97570205819e548536d461ae41fce0241e08
03_limpieza_y_mapeo,data_procesados/registro_unidades.csv,sí,1881,33c3ea91f66091d41d4fd58729da19ac98fd35b498579489367ceb97fb3d7b78
01_inspeccion_BES_jun,reports/figuras/survey_md_tvd_por_hoja.png,sí,89774,e21ea17a8c606802765cc1e3ecd876776c372ab57bea081abb3e87a95c308228
01_inspeccion_BES_jun,reports/figuras/inclinacion_survey_por_hoja.png,sí,24568,ec366dd344a56788c2fe28597bf7ab2243095f297dfac69e96771f2952d5f102
03_mapeo_y_preparacion_BES_jun,data_procesados/curvas_bomba_60hz.csv,sí,1346,baf4dc9a2c8ac8438fbd1e4b51188ee77058f8120af7af71192fd126e73fa70d
03_mapeo_y_preparacion_BES_jun,reports/figuras/head_puntos_60hz.png,sí,59870,b13b8b57e5fe920c2b29317c8c44ac365e53a2d2444d9158026ae5ec4500f392
04_calculos_deterministicos,data_procesados/resultados_deterministicos.csv,sí,2106,7139763b54e2c8688477d3add9485c5cc54fa3b15010589f5219640b1b039613
05_evaluacion_por_pozo,reports/figuras/caudal_limites_por_hoja.png,sí,53464,0cddf853087dd77b6f4c16f4ad831e5fa6479abc558baf1fa2f5ec9e53d387c8
05_evaluacion_por_pozo,reports/figuras/interseccion_sistema_bomba.png,sí,116304,69c256a901d326d4c7ade0304d8079489a40391aa7fe3c0e4557bbab58ab185a


In [6]:
T(pd.DataFrame({"archivo presente sin linaje declarado": huerfanos}))

archivo presente sin linaje declarado
data_procesados/calidad.md
data_procesados/estadisticas_operacion.csv
data_procesados/estadisticas_survey.csv
data_procesados/formulas_muestra.md
data_procesados/perfil_hojas.md
data_procesados/resumen_columnas.csv
data_procesados/resumen_pozos.csv
data_procesados/survey_estaciones.csv
data_procesados/transformaciones.md
reports/figuras/caudal_y_limites.png


**Interpretación.** Los archivos declarados existen y se pueden identificar por huella. La segunda tabla lista lo que hay en las carpetas de salida sin una declaración de linaje: son derivados de las primeras versiones del análisis (tablas descriptivas y figuras de versiones anteriores del código de `src/`) cuyo productor no está declarado en este cuaderno. **No se borran** (una regla de este expediente es no eliminar documentos que no se ha podido demostrar que sobran), pero se registran como pendiente de clasificar en el informe de auditoría. Sobre las figuras, `caudal_y_limites.png` y `head_tabulado.png` son versiones previas de dos de las seis figuras vigentes (`caudal_limites_por_hoja.png` y `head_puntos_60hz.png`).

## Paso 4 · Comparación con el estado previo a la auditoría

**Qué se hace.** Antes de reescribir los cuadernos se guardó en `logs/respaldo_previo_a_auditoria/` una copia de los 12 cuadernos y de los 12 PDF anteriores, con un manifiesto de SHA-256. Se **verifica que ese respaldo sigue íntegro** (cada archivo coincide con su huella) y se compara, cuaderno por cuaderno, el tamaño del original frente al actual. Como no hay Git, esta es la forma de saber qué se cambió y de recuperar un original.

In [7]:
respaldo = RAIZ / "logs" / "respaldo_previo_a_auditoria"
manifiesto = [linea.split(None, 1) for linea in (respaldo / "MANIFIESTO_SHA256.txt").read_text(encoding="utf-8").splitlines() if linea.strip()]
verif = [{"archivo": ruta.strip(), "coincide con el manifiesto": huellas(respaldo / ruta.strip())["sha256"] == sha} for sha, ruta in manifiesto]
integro = pd.DataFrame(verif)
print("Archivos en el manifiesto:", len(integro), "| coinciden:", int(integro["coincide con el manifiesto"].sum()))
comp = []
for nombre, _ in ROLES:
    o = nbformat.read(respaldo / "notebooks" / f"{nombre}.ipynb", as_version=4)
    a = nbformat.read(RAIZ / "notebooks" / f"{nombre}.ipynb", as_version=4)
    pal = lambda nb: sum(len(c.source.split()) for c in nb.cells if c.cell_type == "markdown")
    lin = lambda nb: sum(len(c.source.splitlines()) for c in nb.cells if c.cell_type == "code")
    comp.append({"cuaderno": nombre, "celdas (antes)": len(o.cells), "celdas (ahora)": len(a.cells),
                 "palabras explicativas (antes)": pal(o), "palabras explicativas (ahora)": pal(a),
                 "líneas de código (antes)": lin(o), "líneas de código (ahora)": lin(a)})
comparacion = pd.DataFrame(comp)
T(comparacion, ancha=True)

Archivos en el manifiesto: 24 | coinciden: 24


cuaderno,celdas (antes),celdas (ahora),palabras explicativas (antes),palabras explicativas (ahora),líneas de código (antes),líneas de código (ahora)
00_inventario_y_fuentes,3,25,169,1630,30,125
00_inventario_y_trazabilidad,3,23,120,1467,30,111
01_inspeccion_BES_jun,3,29,209,1971,22,75
01_inspeccion_y_calidad_datos,3,38,378,2218,38,162
02_ecuaciones_y_variables,3,23,203,2854,19,124
02_trazabilidad_y_analisis_formulas,3,32,226,2109,12,171
03_limpieza_y_mapeo,3,35,147,2003,21,134
03_mapeo_y_preparacion_BES_jun,3,32,218,2010,26,90
04_calculos_deterministicos,3,30,232,1563,34,176
04_modelo_deterministico,3,29,133,2385,25,120


**Interpretación.** El respaldo coincide con su manifiesto: los originales están intactos y se pueden recuperar. La comparación muestra el cambio de escala del trabajo: los cuadernos pasaron de **3 celdas** (dos de texto breve y una de código) a documentos con varios pasos, cada uno con su explicación antes y después del código. Lo que **no** cambió es la identidad de cada cuaderno ni su objetivo original (el rol declarado arriba corresponde a lo que el cuaderno ya anunciaba). Si alguna cifra de «ahora» fuera igual o menor que «antes» sería una señal de que el cuaderno no se ha enriquecido y el criterio de revisión lo habría marcado.

## Paso 5 · Criterios de aceptación

In [8]:
crit = Criterios("00_inventario_y_trazabilidad")
crit.comprobar("C00b-1", "El respaldo coincide con su manifiesto", bool(integro["coincide con el manifiesto"].all()) and len(integro) == 24, f"{len(integro)} archivos")
crit.comprobar("C00b-2", "Existen los 12 cuadernos esperados", len(estado) == 12 and bool(estado["existe"].all()), f"{int(estado['existe'].sum())} de 12")
crit.comprobar("C00b-3", "Todo archivo declarado en el linaje existe", bool(linaje["existe"].all()), f"{len(linaje)} archivos")
crit.comprobar("C00b-4", "Los roles de los 12 cuadernos son distintos", len({r for _, r in ROLES}) == 12, "12 roles distintos")
otros = estado[estado["cuaderno"] != "00_inventario_y_trazabilidad"]
crit.comprobar("C00b-5", "Cada cuaderno (salvo éste) tiene Limitaciones y Referencias", bool(otros["con Limitaciones y Referencias"].all()), f"{int(otros['con Limitaciones y Referencias'].sum())} de {len(otros)}")
crit.comprobar("C00b-6", "El libro no cambió", huellas(COPIA_TRABAJO)["sha256"] == sha_inicial, sha_inicial[:16] + "…")
T(crit.tabla())

id,criterio,resultado,evidencia
C00b-1,El respaldo coincide con su manifiesto,CUMPLE,24 archivos
C00b-2,Existen los 12 cuadernos esperados,CUMPLE,12 de 12
C00b-3,Todo archivo declarado en el linaje existe,CUMPLE,11 archivos
C00b-4,Los roles de los 12 cuadernos son distintos,CUMPLE,12 roles distintos
C00b-5,Cada cuaderno (salvo éste) tiene Limitaciones y Referencias,CUMPLE,11 de 11
C00b-6,El libro no cambió,CUMPLE,ccb5a738baea1332…


In [9]:
crit.exigir()
print("Todos los criterios de aceptación del cuaderno 00 (trazabilidad) se cumplen.")

Todos los criterios de aceptación del cuaderno 00 (trazabilidad) se cumplen.


## Conclusión técnica

- **Resultado:** el expediente está inventariado, los 12 cuadernos tienen rol propio (ninguno es duplicado), el linaje de las tablas y figuras vigentes está declarado y verificado por huella, y el estado previo a la auditoría se conserva íntegro con manifiesto.
- **Hallazgo de higiene:** existen archivos de salida sin linaje declarado (versiones previas de tablas y figuras). No se borran; quedan registrados como pendientes de clasificar.
- **Qué habilita:** cualquier cifra o figura puede rastrearse hasta el cuaderno que la produce, y cualquier cuaderno puede compararse con su original.
- **Siguiente paso:** `01_inspeccion_BES_jun`, para describir las hojas.

## Relación con el sistema de reglas

Tampoco formula reglas. Deja escrito cómo se relacionan los doce cuadernos y qué archivo produce cada uno. El motor no lo ejecuta.

## Limitaciones

- El **linaje** (qué cuaderno escribe qué archivo) está **declarado** en una tabla del propio cuaderno; el control verifica existencia y huella, no que el cuaderno indicado haya sido el productor.
- El estado de ejecución se lee de los `.ipynb` guardados en disco, no de una ejecución en vivo; este mismo cuaderno se omite de esa lectura.
- Las huellas de los originales respaldados prueban que no han cambiado desde el respaldo, no que fueran correctos antes.
- No hay control de versiones Git en el proyecto: el diff se hace contra el respaldo con manifiesto.

## Referencias

Solo se listan las fuentes realmente consultadas en este cuaderno.

1. **Libro de trabajo:** `entrada/BES_jun.xlsx`, copia byte a byte de «Copia de FINAL BES-jun.xlsx» (SHA-256 `ccb5a738baea13321993e8b6eccefd6cac90ecda86b1c4b9181f897fcfa83f85`). Cinco hojas: QF-554H, QF-545H, QF-500H, QF-378HST y QF-549HST.
2. **`trabajo.md` y `trabajo2.md`** (raíz del proyecto): plan de trabajo, reglas de no mezclar libros y bloqueo P-14.
3. **Software:** Python 3.14.8, pandas 3.0.6, numpy 2.5.4, matplotlib 3.11.2, openpyxl 3.1.5 (versiones tomadas de la ficha de entorno de cada ejecución).